# Code Generator

The requirement: use a Frontier model to generate high performance C++ code from Python code


<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/resources.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">Reminder: OPTIONAL to execute C++ code or Rust code</h2>
            <span style="color:#f71;">As an alternative, you can run it on the website given yesterday</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h1 style="color:#900;">Important Note</h1>
            <span style="color:#900;">
            In this lab, I use high end models GPT 5, Claude 4.5 Sonnet, Gemini 2.5 Pro, Grok 4, which are the slightly higher priced models. The costs are still low, but if you'd prefer to keep costs ultra low, please pick lower cost models like gpt-5-nano.
            </span>
        </td>
    </tr>
</table>

In [1]:
# imports

import os
import io
import sys
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr
import subprocess
from IPython.display import Markdown, display


In [2]:
load_dotenv(override=True)
#openai_api_key = os.getenv('OPENAI_API_KEY')
#anthropic_api_key = os.getenv('ANTHROPIC_API_KEY')
google_api_key = os.getenv('GOOGLE_API_KEY')
#grok_api_key = os.getenv('GROK_API_KEY')
groq_api_key = os.getenv('GROQ_API_KEY')
#openrouter_api_key = os.getenv('OPENROUTER_API_KEY')

# if openai_api_key:
#     print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
# else:
#     print("OpenAI API Key not set")
    
# if anthropic_api_key:
#     print(f"Anthropic API Key exists and begins {anthropic_api_key[:7]}")
# else:
#     print("Anthropic API Key not set (and this is optional)")

if google_api_key:
    print(f"Google API Key exists and begins {google_api_key[:2]}")
else:
    print("Google API Key not set (and this is optional)")

# if grok_api_key:
#     print(f"Grok API Key exists and begins {grok_api_key[:4]}")
# else:
#     print("Grok API Key not set (and this is optional)")

if groq_api_key:
    print(f"Groq API Key exists and begins {groq_api_key[:4]}")
else:
    print("Groq API Key not set (and this is optional)")

# if openrouter_api_key:
#     print(f"OpenRouter API Key exists and begins {openrouter_api_key[:6]}")
# else:
#     print("OpenRouter API Key not set (and this is optional)")



Google API Key exists and begins AI
Groq API Key exists and begins gsk_


In [3]:
# Connect to client libraries

#openai = OpenAI()

#anthropic_url = "https://api.anthropic.com/v1/"
gemini_url = "https://generativelanguage.googleapis.com/v1beta/openai/"
#grok_url = "https://api.x.ai/v1"
groq_url = "https://api.groq.com/openai/v1"
ollama_url = "http://localhost:11434/v1"
#openrouter_url = "https://openrouter.ai/api/v1"

#anthropic = OpenAI(api_key=anthropic_api_key, base_url=anthropic_url)
gemini = OpenAI(api_key=google_api_key, base_url=gemini_url)
#grok = OpenAI(api_key=grok_api_key, base_url=grok_url)
groq = OpenAI(api_key=groq_api_key, base_url=groq_url)
ollama = OpenAI(api_key="ollama", base_url=ollama_url)
#openrouter = OpenAI(api_key=openrouter_api_key, base_url=openrouter_url)



In [4]:
models = ["gemini-2.5-flash-lite", "qwen2.5-coder", "deepseek-coder-v2", "gpt-oss:20b", "openai/gpt-oss-120b"]
#["gpt-5", "claude-sonnet-4-5-20250929", "grok-4","qwen/qwen3-coder-30b-a3b-instruct",]

clients = {"gemini-2.5-flash-lite": gemini, "openai/gpt-oss-120b": groq, "qwen2.5-coder": ollama, "deepseek-coder-v2": ollama, "gpt-oss:20b": ollama}
#{"gpt-5": openai, "claude-sonnet-4-5-20250929": anthropic, "grok-4": grok, "qwen/qwen3-coder-30b-a3b-instruct": openrouter}

# Want to keep costs ultra-low? Replace this with models of your choice, using the examples from yesterday

In [5]:
from system_info import retrieve_system_info, rust_toolchain_info

system_info = retrieve_system_info()
rust_info = rust_toolchain_info()
rust_info

{'installed': True,
 'rustc': {'path': '/home/cristian/.cargo/bin/rustc',
  'version': 'rustc 1.93.1 (01f6ddf75 2026-02-11)',
  'host_triple': 'x86_64-unknown-linux-gnu',
  'release': '1.93.1',
  'commit_hash': '01f6ddf7588f42ae2d7eb0a2f21d44e8e96674cf'},
 'cargo': {'path': '/home/cristian/.cargo/bin/cargo',
  'version': 'cargo 1.93.1 (083ac5135 2025-12-15)'},
 'rustup': {'path': '/home/cristian/.cargo/bin/rustup',
  'version': 'rustup 1.28.2 (e4f3ad6f8 2025-04-28)',
  'active_toolchain': 'stable-x86_64-unknown-linux-gnu (default)',
  'default_toolchain': '',
  'toolchains': ['stable-x86_64-unknown-linux-gnu (active, default)'],
  'targets_installed': ['x86_64-unknown-linux-gnu']},
 'rust_analyzer': {'path': '/home/cristian/.cargo/bin/rust-analyzer'},
 'env': {'CARGO_HOME': '/home/cristian/.cargo',
  'RUSTUP_HOME': '/home/cristian/.rustup',
  'RUSTFLAGS': '',
  'CARGO_BUILD_TARGET': ''},
 'execution_examples': ['"/home/cristian/.cargo/bin/cargo" build',
  '"/home/cristian/.cargo/bin/ca

In [6]:
message = f"""
Here is a report of the system information for my computer.
I want to run a Rust compiler to compile a single rust file called main.rs and then execute it in the simplest way possible.
Please reply with whether I need to install a Rust toolchain to do this. If so, please provide the simplest step by step instructions to do so.

If I'm already set up to compile Rust code, then I'd like to run something like this in Python to compile and execute the code:
```python
compile_command = # something here - to achieve the fastest possible runtime performance
compile_result = subprocess.run(compile_command, check=True, text=True, capture_output=True)
run_command = # something here
run_result = subprocess.run(run_command, check=True, text=True, capture_output=True)
return run_result.stdout
```
Please tell me exactly what I should use for the compile_command and run_command.
Have the maximum possible runtime performance in mind; compile time can be slow. Fastest possible runtime performance for this platform is key.
Reply with the commands in markdown.

System information:
{system_info}

Rust toolchain information:
{rust_info}
"""

response = groq.chat.completions.create(model=models[4], messages=[{"role": "user", "content": message}])
display(Markdown(response.choices[0].message.content))

**Do you need to install a Rust tool‑chain?**  
No – the system already has a working Rust tool‑chain:

```
rustc 1.93.1 (01f6ddf75 2026-02-11)
cargo 1.93.1
rustup 1.28.2
```

You can compile and run a single‑file program (`main.rs`) directly with `rustc`.

---

## Fastest possible *runtime* performance  

For a one‑off compilation the best runtime speed is obtained by:

* maximum optimisations (`-C opt-level=3`)
* generating code for the exact host CPU (`-C target-cpu=native`)
* enabling link‑time optimisation (`-C lto=yes`)
* using a single code‑gen unit (helps the optimiser) (`-C codegen-units=1`)
* aborting on panic (removes the unwind tables) (`-C panic=abort`)

Putting those flags together gives the “speed‑only” build.

### 1. Compile command (`compile_command`)

```python
compile_command = [
    "/home/cristian/.cargo/bin/rustc",   # rustc binary
    "main.rs",                           # source file
    "-O",                                # same as -C opt-level=3
    "-C", "target-cpu=native",
    "-C", "lto=yes",
    "-C", "codegen-units=1",
    "-C", "panic=abort",
    "-C", "debuginfo=0",                # no debug info → smaller binary
    "-o", "main_binary"                  # name of the produced executable
]
```

*Explanation*  

* `-O` is a shortcut for `-C opt-level=3`.  
* All `-C …` options are passed directly to the LLVM backend.  
* The output file (`-o main_binary`) can be any name you like – here we use `main_binary` to avoid a clash with the source file.

### 2. Run command (`run_command`)

```python
run_command = ["./main_binary"]
```

Make sure the binary is executable (the `rustc` command creates it with the proper permission, but if you ever copy it you can run `chmod +x main_binary`).

---

## Full Python example

```python
import subprocess
import pathlib

# ----------------------------------------------------------------------
# 1️⃣  Compile
# ----------------------------------------------------------------------
compile_command = [
    "/home/cristian/.cargo/bin/rustc",
    "main.rs",
    "-O",
    "-C", "target-cpu=native",
    "-C", "lto=yes",
    "-C", "codegen-units=1",
    "-C", "panic=abort",
    "-C", "debuginfo=0",
    "-o", "main_binary"
]

compile_result = subprocess.run(
    compile_command,
    check=True,
    text=True,
    capture_output=True
)

print("Compilation stdout:", compile_result.stdout)
print("Compilation stderr:", compile_result.stderr)

# ----------------------------------------------------------------------
# 2️⃣  Execute
# ----------------------------------------------------------------------
run_command = ["./main_binary"]

run_result = subprocess.run(
    run_command,
    check=True,
    text=True,
    capture_output=True
)

print("Program output:", run_result.stdout)
```

Running the above script will:

1. Compile `main.rs` with the highest‑performance settings.
2. Execute the resulting binary and return its standard output.

---

### Quick “install Rust” cheat‑sheet (if you ever need it)

If you ever end up on a machine **without** Rust, the quickest way is:

```bash
# 1. Install required system packages
sudo apt update && sudo apt install -y curl build-essential

# 2. Install rustup (the official installer) – it adds rustc, cargo, rustup
curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y

# 3. Ensure the binaries are in your PATH (restart the shell or source)
source "$HOME/.cargo/env"
```

That will give you the same `rustc`/`cargo` you already have.

---

**Bottom line:** you are already set up. Use the `compile_command` and `run_command` shown above for the fastest possible runtime performance on your Debian 12 (x86_64) system.

## For C++, overwrite this with the commands from yesterday, or for Rust, use the new commands

Or just use the website like yesterday:

 https://www.programiz.com/cpp-programming/online-compiler/

In [7]:
compile_command = [
    "/Users/ed/.cargo/bin/rustc",
    "main.rs",
    "-C", "opt-level=3",
    "-C", "target-cpu=native",
    "-C", "codegen-units=1",
    "-C", "lto=fat",
    "-C", "panic=abort",
    "-C", "strip=symbols",
    "-o", "main",
]

run_command = ["./main"]


## And now, on with the main task

In [8]:
language = "Rust" # or "C++"
extension = "rs" if language == "Rust" else "cpp"

system_prompt = f"""
Your task is to convert Python code into high performance {language} code.
Respond only with {language} code. Do not provide any explanation other than occasional comments.
The {language} response needs to produce an identical output in the fastest possible time.
"""

def user_prompt_for(python):
    return f"""
Port this Python code to {language} with the fastest possible implementation that produces identical output in the least time.
The system information is:
{system_info}
Your response will be written to a file called main.{language} and then compiled and executed; the compilation command is:
{compile_command}
Respond only with {language} code.
Python code to port:

```python
{python}
```
"""

In [9]:
def messages_for(python):
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt_for(python)}
    ]
 

In [10]:
def write_output(code):
    with open(f"main.{extension}", "w") as f:
        f.write(code)

In [11]:
def port(model, python):
    client = clients[model]
    reasoning_effort = "high" if 'gpt' in model else None
    response = client.chat.completions.create(model=model, messages=messages_for(python), reasoning_effort=reasoning_effort)
    reply = response.choices[0].message.content
    reply = reply.replace('```cpp','').replace('```rust','').replace('```','')
    return reply

In [12]:
def run_python(code):
    globals_dict = {"__builtins__": __builtins__}

    buffer = io.StringIO()
    old_stdout = sys.stdout
    sys.stdout = buffer

    try:
        exec(code, globals_dict)
        output = buffer.getvalue()
    except Exception as e:
        output = f"Error: {e}"
    finally:
        sys.stdout = old_stdout

    return output

In [13]:
# Use the commands from GPT 5

def compile_and_run(code):
    write_output(code)
    try:
        subprocess.run(compile_command, check=True, text=True, capture_output=True)
        run_result = subprocess.run(run_command, check=True, text=True, capture_output=True)
        return run_result.stdout
    except subprocess.CalledProcessError as e:
        return f"An error occurred:\n{e.stderr}"

In [14]:
python_hard = """# Be careful to support large numbers

def lcg(seed, a=1664525, c=1013904223, m=2**32):
    value = seed
    while True:
        value = (a * value + c) % m
        yield value
        
def max_subarray_sum(n, seed, min_val, max_val):
    lcg_gen = lcg(seed)
    random_numbers = [next(lcg_gen) % (max_val - min_val + 1) + min_val for _ in range(n)]
    max_sum = float('-inf')
    for i in range(n):
        current_sum = 0
        for j in range(i, n):
            current_sum += random_numbers[j]
            if current_sum > max_sum:
                max_sum = current_sum
    return max_sum

def total_max_subarray_sum(n, initial_seed, min_val, max_val):
    total_sum = 0
    lcg_gen = lcg(initial_seed)
    for _ in range(20):
        seed = next(lcg_gen)
        total_sum += max_subarray_sum(n, seed, min_val, max_val)
    return total_sum

# Parameters
n = 10000         # Number of random numbers
initial_seed = 42 # Initial seed for the LCG
min_val = -10     # Minimum value of random numbers
max_val = 10      # Maximum value of random numbers

# Timing the function
import time
start_time = time.time()
result = total_max_subarray_sum(n, initial_seed, min_val, max_val)
end_time = time.time()

print("Total Maximum Subarray Sum (20 runs):", result)
print("Execution Time: {:.6f} seconds".format(end_time - start_time))
"""

In [17]:
from styles import CSS

with gr.Blocks(css=CSS, theme=gr.themes.Monochrome(), title=f"Port from Python to {language}") as ui:
    with gr.Row(equal_height=True):
        with gr.Column(scale=6):
            python = gr.Code(
                label="Python (original)",
                value=python_hard,
                language="python",
                lines=26
            )
        with gr.Column(scale=6):
            cpp = gr.Code(
                label=f"{language} (generated)",
                value="",
                language="cpp",
                lines=26
            )

    with gr.Row(elem_classes=["controls"]):
        python_run = gr.Button("Run Python", elem_classes=["run-btn", "py"])
        model = gr.Dropdown(models, value=models[0], show_label=False)
        convert = gr.Button(f"Port to {language}", elem_classes=["convert-btn"])
        cpp_run = gr.Button(f"Run {language}", elem_classes=["run-btn", "cpp"])

    with gr.Row(equal_height=True):
        with gr.Column(scale=6):
            python_out = gr.TextArea(label="Python result", lines=8, elem_classes=["py-out"])
        with gr.Column(scale=6):
            cpp_out = gr.TextArea(label=f"{language} result", lines=8, elem_classes=["cpp-out"])

    convert.click(fn=port, inputs=[model, python], outputs=[cpp])
    python_run.click(fn=run_python, inputs=[python], outputs=[python_out])
    cpp_run.click(fn=compile_and_run, inputs=[cpp], outputs=[cpp_out])

ui.launch(share=True)


Traceback (most recent call last):
  File "/home/cristian/projects/llm_engineering/.venv/lib/python3.12/site-packages/gradio/queueing.py", line 759, in process_events
    response = await route_utils.call_process_api(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/cristian/projects/llm_engineering/.venv/lib/python3.12/site-packages/gradio/route_utils.py", line 354, in call_process_api
    output = await app.get_blocks().process_api(
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/cristian/projects/llm_engineering/.venv/lib/python3.12/site-packages/gradio/blocks.py", line 2116, in process_api
    result = await self.call_function(
             ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/cristian/projects/llm_engineering/.venv/lib/python3.12/site-packages/gradio/blocks.py", line 1623, in call_function
    prediction = await anyio.to_thread.run_sync(  # type: ignore
                 ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/cristian/project

## RESULTS!

Qwen 2.5 Coder: FAIL  
Gemini 2.5 Pro: FAIL  
DeepSeek Coder v2: FAIL  
Qwen3 Coder 30B: FAIL  
Claude Sonnet 4.5: FAIL    
GPT-5: FAIL    

3rd place: GPT-oss-20B: 0.000341  
2nd place: Grok 4: 0.000317  
**1st place: OpenAI GPT-OSS 120B: 0.000304**  

In [ ]:
print(f"In Ed's experimenet, the GPT-OSS 120B model outcome is {33.755209/0.000304:,.0f} times faster than the Python code.")